# Functions  

## Function Definition

  * Functions can be named or anonymous. 
  * Result of the last expression in the function is returned. The `return` keyword can be used to explicitly return at the end of the function but that is not usually done. It is used in the following situations -
    - To short circuit a function
    - To return "void" by having a `return` statement without any expression after it.
    - To return "void" by `return nothing`
  * Anonymous functions are what are called lambdas in other languages, are called "closures" in Julia.
  * Can return tuples
  * Arguments are passed as pointers, so reference types like arrays are susceptible to changes.

In [1]:
# Standard style function declaration
# No need for a return statement. The last expression is returned.
function hello(name)
    greeting = "Namaste"
    say = string(greeting, " ", name, ". ")
    string(say, "Its good to see you!")
end

hello (generic function with 1 method)

In [2]:
hello("APTG")

"Namaste APTG. Its good to see you!"

In [3]:
# But return statements are fine
function addpos(x, y)
    if x <= 0 || y <= 0
        return nothing
    end
    x + y
end

addpos (generic function with 1 method)

In [4]:
addpos(1, 1)

2

In [5]:
addpos(1, -1)

In [6]:
# Single line function definitions
add_2(x) = x + 2

add_2 (generic function with 1 method)

In [7]:
add_2(40)

42

In [8]:
# Anonymous function
op = x -> x + 2

#23 (generic function with 1 method)

In [9]:
# Another way to specify an anonymous function
op2 = function (x)
    x^2 + 2x - 1
end

#26 (generic function with 1 method)

In [10]:
op(40)

42

In [11]:
op2(40)

1679

In [12]:
typeof(op)

var"#23#24"

In [13]:
typeof(op2)

var"#26#27"

In [14]:
typeof(hello)

typeof(hello) (singleton type of function hello, subtype of Function)

In [15]:
typeof(add_2)

typeof(add_2) (singleton type of function add_2, subtype of Function)

In [16]:
function f(a, b)
    a + b, a * b
end

x, y = f(10, 20)
print(x, " ", y)

30 200

In [17]:
function mutate(ary)
    ary[1] = "HAHA"
end

mutate (generic function with 1 method)

In [18]:
tp = ["Hello", "World"]
# The return value will be the first element of ary
mutate(tp)
tp

2-element Vector{String}:
 "HAHA"
 "World"

## Functions Style Guide

### Naming Conventions
  * Append `!` to functions that modify their arguments, except functions related to IO or RNGs. E.g., `sort` and `sort!`, `read(io)` (which only modifies the IO so no bang), and `read!(io, x)` which modifies `x` as well so bang.

  * Functions are lowercase and, when readable, with multiple words squashed together (`isequal`, `haskey`). Use snake_case only when necessary. 

  * Conciseness is valued, but avoid abbreviation (`indexin` instead of `idxin`).

  * If a function name requires multiple words, consider whether it might represent more than one concept and might be better split into pieces.

### Parameter Ordering
Parameters should be passed to a function in the following order -
  1. Function pointers
  2. I/O stream
  3. Mutating input
  4. Type
  5. Non-mutating input
  6. Key
  7. Value
  8. Everything else
  9. Varargs (covered below)
  10. Keyword args (covered below)

## Parameter Types

  * Input parameters can have type annotations and the general practice of not being too restrictive with the types holds.
  * Return types can be also be annotated as part of the function definition, but this is rarely seen in Julia.
  * Un-annotated functions are duck-typed.

In [19]:
# Not using a concrete type like Int8 or Int16 or even Int or anything like that. 
# Using the abstract type Integer
function bake(ncookies::Integer)
    println("Baking $ncookies right now.")
end

bake (generic function with 1 method)

In [20]:
Int <: Integer

true

In [21]:
Int8 <: Int

false

Methods with type annotated parameters are contravariant like in every other language, i.e., I can call a function that takes in base type parameters, with subtype arguments. In the example below, `Int8` and `Int16` are subtypes of `Integer`, but `Float16` is not.

In [2]:
function add(x::Integer, y::Integer)
    println("typeof(x) = $(typeof(x)), typeof(y) = $(typeof(y))")
    x + y
end

add (generic function with 1 method)

In [4]:
a::Int8 = 1
b::Int8 = 2
add(a, b)

typeof(x) = Int8, typeof(y) = Int8


3

In [5]:
u::Int16 = 1
v::Int16 = 2
add(u, v)

typeof(x) = Int16, typeof(y) = Int16


3

In [6]:
p::Float16 = 1.1
q::Float16 = 2.2
add(p, q)

MethodError: MethodError: no method matching add(::Float16, ::Float16)
The function `add` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  add(!Matched::Integer, !Matched::Integer)
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y250sZmlsZQ==.jl:1


### Duck Typing
The same function can be called with different argument types. It will work as long as the all the functions or operators used inside the function are supported for the given argument types. E.g., the function defined below will work for any input type that supports the power-of operator. This includes numeric types like scalar integers and even matrices. But it also includes strings which implements the power-of op as a repeater. But this will not work on the Vector type.

In [22]:
function square_or_cube(x)
    if rand() > 0.5
        x = x^2
    else
        x = x^3
    end
    x
end

square_or_cube (generic function with 1 method)

In [23]:
for _ in 1:10
    println(square_or_cube(4))
end

64
64
64
64
64
16
16
16
64
16


In [24]:
# A pointer to u is passed to the function. Inside the function I am re-assigning
# the pointer so it points to a different memory locaiton and the memory location 
# in this cell is left alone and unchanged.
u = 3
v = square_or_cube(u)
println("u = $u")
println("v = $v")

u = 3
v = 27


In [25]:
U = [1 2; 3 4]
square_or_cube(U)

2×2 Matrix{Int64}:
 37   54
 81  118

In [26]:
square_or_cube("APTG")

"APTGAPTGAPTG"

In [27]:
v = rand(3)

3-element Vector{Float64}:
 0.7777411186035902
 0.13406969085802567
 0.4344580136592593

In [28]:
try
    square_or_cube(v)
catch e
    @assert e isa MethodError
    println("Got MethodError")
end

Got MethodError


## Multiple Dispatch

This applies to function overloading. If I have multiple functions with the same name, the compiler or the runtime needs to figure out which function to call. If it uses the type of a single argument to determine this, it is called **single dispatch**. In OOP languages, this argument is usually the implict `this` argument, i.e., the object type on which the method is being called determines which method to invoke. Python also has a `@functools.singledispatch` decorator which uses the type of the first argument to figure out which function to invoke. This is how a user can implement a function like `len` on different types.

In contrast, Julia has multiple dispatch, where the types of **all** the arguments are used to determine which function to call. The way Julia implements this by having a function object (maybe a struct?) with multiple methods defined on it. In Julialand the name is called "generic function" and the type-specific implementations are called "methods".

If I don't specify the function parameter types, then this is a "generic function" and I can use duck typing to call this function with different argument types.

In [29]:
show(x, y) = println("I was called $x and $y")

show (generic function with 1 method)

In [30]:
typeof(show)

typeof(show) (singleton type of function show, subtype of Function)

In [31]:
show("Avilay", "Parekh")

I was called Avilay and Parekh


In [32]:
show(1, 2)

I was called 1 and 2


However, if I do specify the parameter types, then I can only call this function with the specified types. But the benefit of doing this is that I can define different implementation for different input parameter types. Unlike pattern matching in Haskell, the order of function declaration are not important. Below I define the most expansive types first and more specific types later. The Julia runtime will still be able to dispatch the function calls correctly.

In [33]:
# repr(x::Any, y::Any) = println("Generic repr - $x $y")
repr(x, y) = println("Generic repr - $x $y")
repr(x::String, y::String) = println("I was called with $x and $y")
repr(x::Number, y::Number) = println("My input values are - $x and $y")

repr (generic function with 3 methods)

In [34]:
typeof(repr)

typeof(repr) (singleton type of function repr, subtype of Function)

In [35]:
repr("Avilay", "Parekh")

I was called with Avilay and Parekh


In [36]:
repr(1, 2)

My input values are - 1 and 2


In [37]:
repr(map, broadcast)

Generic repr - map broadcast


I can figure out all the methods defined on a function name using the `methods` function. For example, the `+` function is pretty heavily overloaded. Lets examine all the "methods" defined on it.

In [38]:
methods(+)

# 191 methods for generic function "+" from Base:
   [1] +(::Missing, ::Missing)
     @ missing.jl:122
   [2] +(::Missing)
     @ missing.jl:101
   [3] +(x::Missing, y::Dates.AbstractTime)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:91
   [4] +(::Missing, ::Number)
     @ missing.jl:123
   [5] +(x::Bool, z::Complex{Bool})
     @ complex.jl:308
   [6] +(x::Bool, y::Bool)
     @ bool.jl:168
   [7] +(x::Bool)
     @ bool.jl:165
   [8] +(x::Bool, z::Complex)
     @ complex.jl:315
   [9] +(x::Bool, y::T) where T<:AbstractFloat
     @ bool.jl:175
  [10] +(dt::Dates.Date, t::Dates.Time)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:21
  [11] +(dt::Dates.Date, y::Dates.Year)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:29
  [12] +(dt::Dates.Date, z::Dates.Month)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:56
  [13] +(x::Dates.Date, y::Dates.Quarter)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:75
  [14] +(x::Dates.Date, y::Dates.Week)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:79
  [15] +(x::Dates.Date, y::Dates.Day)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:81
  [16] +(level::Base.CoreLogging.LogLevel, inc::Integer)
     @ logging/logging.jl:132
  [17] +(t::Dates.Time, dt::Dates.Date)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:22
  [18] +(x::Dates.Time, y::Dates.TimePeriod)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:85
  [19] +(x::Dates.CompoundPeriod, y::Dates.CompoundPeriod)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/periods.jl:335
  [20] +(x::Dates.CompoundPeriod, y::Dates.Period)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/periods.jl:333
  [21] +(x::Dates.CompoundPeriod, y::Dates.TimeType)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/periods.jl:363
  [22] +(a::Pkg.Resolve.VersionWeight, b::Pkg.Resolve.VersionWeight)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Pkg/src/Resolve/versionweights.jl:22
  [23] +(B::BitMatrix, J::LinearAlgebra.UniformScaling)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/uniformscaling.jl:154
  [24] +(x::BigInt, y::BigInt)
     @ gmp.jl:502
  [25] +(a::BigInt, b::BigInt, c::BigInt)
     @ gmp.jl:542
  [26] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt)
     @ gmp.jl:543
  [27] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt, e::BigInt)
     @ gmp.jl:544
  [28] +(x::BigInt, y::BigInt, rest::BigInt...)
     @ gmp.jl:679
  [29] +(c::BigInt, x::BigFloat)
     @ mpfr.jl:613
  [30] +(x::BigInt, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ gmp.jl:550
  [31] +(x::BigInt, c::Union{Int16, Int32, Int64, Int8})
     @ gmp.jl:556
  [32] +(a::Pkg.Resolve.FieldValue, b::Pkg.Resolve.FieldValue)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Pkg/src/Resolve/fieldvalues.jl:43
  [33] +(x::BigFloat, c::BigInt)
     @ mpfr.jl:608
  [34] +(a::BigFloat, b::BigFloat, c::BigFloat, d::BigFloat, e::BigFloat)
     @ mpfr.jl:764
  [35] +(x::BigFloat, y::BigFloat)
     @ mpfr.jl:577
  [36] +(a::BigFloat, b::BigFloat, c::BigFloat)
     @ mpfr.jl:751
  [37] +(a::BigFloat, b::BigFloat, c::BigFloat, d::BigFloat)
     @ mpfr.jl:757
  [38] +(x::BigFloat, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ mpfr.jl:584
  [39] +(x::BigFloat, c::Union{Int16, Int32, Int64, Int8})
     @ mpfr.jl:592
  [40] +(x::BigFloat, c::Union{Float16, Float32, Float64})
     @ mpfr.jl:600
  [41] +(dt::Dates.D

There are 191 methods defined on this function! When I call a function, I can use the `@which` macro to figure out which method was actually called.

In [39]:
@which +(2, 3)

+(x::T, y::T) where T<:Union{Int128, Int16, Int32, Int64, Int8, UInt128, UInt16, UInt32, UInt64, UInt8}
     @ Base int.jl:87

In [40]:
@which +(2.7, 3.14)

+(x::T, y::T) where T<:Union{Float16, Float32, Float64}
     @ Base float.jl:495

In [41]:
@which +("Hello", "World")

ErrorException: Calling invoke(f, t, args...) would throw:
MethodError: no method matching invoke +(::String, ::String)
The function `+` exists, but no method is defined for this combination of argument types.
String concatenation is performed with * (See also: https://docs.julialang.org/en/v1/manual/strings/#man-concatenation).

Closest candidates are:
  +(::Any, ::Any, !Matched::Any, !Matched::Any...)
   @ Base operators.jl:642
  +(!Matched::Missing, !Matched::Missing)
   @ Base missing.jl:122
  +(!Matched::Missing)
   @ Base missing.jl:101
  ...


As can be seen above, there is no method defined for two strings as input. Lets define our own method for this.

In [42]:
import Base: +

In [43]:
+(x::String, y::String) = string(x, " ", y)

+ (generic function with 192 methods)

In [44]:
"Hello" + "World"

"Hello World"

Now if I run `methods` I should see 192 methods. 

In [45]:
methods(+)

# 192 methods for generic function "+" from Base:
   [1] +(::Missing, ::Missing)
     @ missing.jl:122
   [2] +(::Missing)
     @ missing.jl:101
   [3] +(x::Missing, y::Dates.AbstractTime)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:91
   [4] +(::Missing, ::Number)
     @ missing.jl:123
   [5] +(x::Bool, z::Complex{Bool})
     @ complex.jl:308
   [6] +(x::Bool, y::Bool)
     @ bool.jl:168
   [7] +(x::Bool)
     @ bool.jl:165
   [8] +(x::Bool, z::Complex)
     @ complex.jl:315
   [9] +(x::Bool, y::T) where T<:AbstractFloat
     @ bool.jl:175
  [10] +(dt::Dates.Date, t::Dates.Time)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:21
  [11] +(dt::Dates.Date, y::Dates.Year)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:29
  [12] +(dt::Dates.Date, z::Dates.Month)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:56
  [13] +(x::Dates.Date, y::Dates.Quarter)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:75
  [14] +(x::Dates.Date, y::Dates.Week)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:79
  [15] +(x::Dates.Date, y::Dates.Day)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:81
  [16] +(level::Base.CoreLogging.LogLevel, inc::Integer)
     @ logging/logging.jl:132
  [17] +(t::Dates.Time, dt::Dates.Date)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:22
  [18] +(x::Dates.Time, y::Dates.TimePeriod)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/arithmetic.jl:85
  [19] +(x::Dates.CompoundPeriod, y::Dates.CompoundPeriod)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/periods.jl:335
  [20] +(x::Dates.CompoundPeriod, y::Dates.Period)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/periods.jl:333
  [21] +(x::Dates.CompoundPeriod, y::Dates.TimeType)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Dates/src/periods.jl:363
  [22] +(a::Pkg.Resolve.VersionWeight, b::Pkg.Resolve.VersionWeight)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Pkg/src/Resolve/versionweights.jl:22
  [23] +(x::String, y::String)
     @ ~/projects/github/learn-langs/learn-julia/IntroToJuliaCourse/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y400sZmlsZQ==.jl:1
  [24] +(B::BitMatrix, J::LinearAlgebra.UniformScaling)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/uniformscaling.jl:154
  [25] +(x::BigInt, y::BigInt)
     @ gmp.jl:502
  [26] +(a::BigInt, b::BigInt, c::BigInt)
     @ gmp.jl:542
  [27] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt)
     @ gmp.jl:543
  [28] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt, e::BigInt)
     @ gmp.jl:544
  [29] +(x::BigInt, y::BigInt, rest::BigInt...)
     @ gmp.jl:679
  [30] +(c::BigInt, x::BigFloat)
     @ mpfr.jl:613
  [31] +(x::BigInt, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ gmp.jl:550
  [32] +(x::BigInt, c::Union{Int16, Int32, Int64, Int8})
     @ gmp.jl:556
  [33] +(a::Pkg.Resolve.FieldValue, b::Pkg.Resolve.FieldValue)
     @ ~/.julia/juliaup/julia-1.12.1+0.x64.linux.gnu/share/julia/stdlib/v1.12/Pkg/src/Resolve/fieldvalues.jl:43
  [34] +(x::BigFloat, c::BigInt)
     @ mpfr.jl:608
  [35] +(a::BigFloat, b::BigFloat, c::BigFloat, d::BigFloat, e::BigFloat)
     @ mpfr.jl:764
  [36] +(x::BigFloat, y::BigFloat)
     @ mpfr.jl:577
  [37] +(a::BigFloat, b::BigFloat, c::BigFloat)
     @ mpfr.jl:751
  [38] +(a::BigFloat, b::BigFloat, c::BigFloat, d::BigFloat)
     @ mpfr.jl:757
  [39] +(x::BigFloat, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ mpfr.jl:584
  [40] +

In [46]:
@which "Hello" + "World"

+(x::String, y::String)
     @ Main ~/projects/github/learn-langs/learn-julia/IntroToJuliaCourse/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y400sZmlsZQ==.jl:1

## Parametric Methods

Ref: https://docs.julialang.org/en/v1/manual/methods/#Parametric-Methods

The syntax is similar to the `UnionAll` type (see Types.ipynb). The gist is that the `where {T}` iterates over all possible values of `T`, which is all types. I can constraint the type with either a lower bound or an upper bound with `where {T<:Number}` s.t `T` will only take on types that are subtypes of `Number`, or even with a lower bound `where {T>:Int}` will take on all the ancestors of `Int` (aka `Int64` on 64 bit systems). 

In [47]:
same_type(x::T, y::T) where {T} = true
same_type(x, y) = false

same_type (generic function with 2 methods)

In [48]:
same_type(1, 2)

true

In [49]:
same_type(1, "hahaha")

false

In [50]:
same_type(Int32(1), Int64(2))

false

In [51]:
function prepend(v::Vector{T}, x::T) where {T}
    [x, v...]
end

prepend (generic function with 1 method)

In [52]:
prepend([1, 2, 3], 0)

4-element Vector{Int64}:
 0
 1
 2
 3

In [53]:
try
    prepend([1, 2, 3], 1.1)
catch e
    @assert e isa MethodError
    println("Got MethodError")
end

Got MethodError


The below example will take in any two numbers **as long as they are of the same type** and return `true`, if the arguments are numbers but not of the same type, the second method kicks in and returns a `false`. There is no method defined for non-numeric arguments.

In [54]:
same_type_numeric(x::T, y::T) where {T<:Number} = true
same_type_numeric(x::Number, y::Number) = false

same_type_numeric (generic function with 2 methods)

In [55]:
same_type_numeric(1, 2)

true

In [56]:
same_type_numeric(1, 1.1)

false

In [ ]:
try
    same_type_numeric(1, "hahaha")
catch e
    @assert e isa MethodError
    println("Got MethodError")
end

Got MethodError


💣 The following example tries to demo lower bound parametric methods, but I haven't gotten it to work yet.

In [14]:
function lowerbound(x::T, y::T) where {T>:Int64}
    println("typeof(x) = $(typeof(x)), typeof(y) = $(typeof(y))")
    true
end


function lowerbound(x::Number, y::Number)
    println("typeof(x) = $(typeof(x)), typeof(y) = $(typeof(y))")
    false
end

function lowerbound(x, y)
    println("typeof(x) = $(typeof(x)), typeof(y) = $(typeof(y))")
    error("Not allowed!")
end

lowerbound (generic function with 3 methods)

In [15]:
a1::Int = 1
b1::Int = 2
lowerbound(a1, b1)

typeof(x) = Int64, typeof(y) = Int64


false

## Optional Parameters

If the function declaration has default values for some parameters then those parameters are optional. Optional parameters must be the last parameters. Optional parameters are a concept of the top level "generic" function, not of individual type-specific methods.

In [58]:
f(a=1, b=2) = a + 2b
f(a::Int, b::Int) = a - 2b

# translates to the following four methods
# f(a, b) = a + 2b
# f(a::Int, b::Int) = a - 2b 
# f(a) = f(a, 2) or f(a, 2.2) or ... depending on the type of a
# f() = f(1, 2) = f(1::Int, 2::Int) = 1 - 2*2 = -3

f (generic function with 4 methods)

In [59]:
f(1.2, 3.4)

8.0

In [60]:
f(1, 3)

-5

In [61]:
f(1)

-3

In [62]:
f(1.1)

5.1

In [63]:
f()

-3

In [64]:
function hello(greeting="Hello", name)
    println(greeting, " ", name)
end

ErrorException: syntax: optional positional arguments must occur at end around /home/avilay/projects/github/learn-langs/learn-julia/IntroToJuliaCourse/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y423sZmlsZQ==.jl:1

In [65]:
function hello(name, greeting="Hello")
    println(greeting, " ", name)
end

hello (generic function with 2 methods)

In [66]:
hello("APTG")

Hello APTG


In [67]:
hello("APTG", "Namaste")

Namaste APTG


## Splatting

In their simplest form, tuples can be splatted into positional arguments when calling a function with multiple parameters. The ellipses `...` is used to splat a tuple into arguments when calling a function.

In [68]:
function splat(x1, x2, x3, x4)
    x1 + x2 + x3 + x4
end

splat (generic function with 1 method)

In [69]:
xs = (10, 20, 30)
splat(1, xs...)

61

## Varargs

  * Varargs are declared using the ellipses `...` in the function signature.
  * Varargs are collected inside a tuple in the function body.
  * Varargs are optional, if none are provided the tuple inside the function body will be an empty tuple.
  * Can only occur at the end.
  * I can specify the type of varargs using the `Vararg{T}` type.
  * There is also a `Vararg{T, N}` type, this lets me specify exactly how many arguments of this type are allowed. 
  > 🤔 This seems a bit silly because if I already know how many args, why am I modeling it as variable number of args?
  * Expanded splatted arguments can be used to fill in positional arguments, followed by varargs.

In [70]:
function varargs(x, y, z...)
    println("x = $x, typeof(x) = $(typeof(x))")
    println("y = $y, typeof(y) = $(typeof(y))")
    println("z = $z, typeof(z) = $(typeof(z))")
end

varargs (generic function with 1 method)

In [71]:
varargs(1, 2, 3, 4)

x = 1, typeof(x) = Int64
y = 2, typeof(y) = Int64
z = (3, 4), typeof(z) = Tuple{Int64, Int64}


In [72]:
varargs(1, 2)

x = 1, typeof(x) = Int64
y = 2, typeof(y) = Int64
z = (), typeof(z) = Tuple{}


In [73]:
# The first element of the splatted tuple is used for the second positional argument.
# The rest of the elements are used to fill in the varargs.
varargs(1, xs...)

x = 1, typeof(x) = Int64
y = 10, typeof(y) = Int64
z = (20, 30), typeof(z) = Tuple{Int64, Int64}


In [16]:
function h(x::Number, ys::Vararg{Number})
    println("x = $x, typeof(x) = $(typeof(x))")
    for y in ys
        println("y = $y, typeof(y) = $(typeof(y))")
    end
end

h (generic function with 1 method)

In [17]:
h(1)

x = 1, typeof(x) = Int64


In [18]:
h(1, 2)

x = 1, typeof(x) = Int64
y = 2, typeof(y) = Int64


In [19]:
h(1, 2, 3, 4, 5)

x = 1, typeof(x) = Int64
y = 2, typeof(y) = Int64
y = 3, typeof(y) = Int64
y = 4, typeof(y) = Int64
y = 5, typeof(y) = Int64


In [74]:
# Can only call this function with exactly 3 arguments
function g(x::Number, ys::Vararg{Number,2})
    println("x = $x, typeof(x) = $(typeof(x))")
    for y in ys
        println("y = $y, typeof(y) = $(typeof(y))")
    end
end

g (generic function with 1 method)

In [75]:
g(1, 2)

MethodError: MethodError: no method matching g(::Int64, ::Int64)
The function `g` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  g(::Number, ::Number, !Matched::Number)
   @ Main ~/projects/github/learn-langs/learn-julia/IntroToJuliaCourse/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y454sZmlsZQ==.jl:2


In [76]:
g(1, 2, 3)

x = 1, typeof(x) = Int64
y = 2, typeof(y) = Int64
y = 3, typeof(y) = Int64


In [77]:
g(1, 2, 3, 4)

MethodError: MethodError: no method matching g(::Int64, ::Int64, ::Int64, ::Int64)
The function `g` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  g(::Number, ::Number, ::Number)
   @ Main ~/projects/github/learn-langs/learn-julia/IntroToJuliaCourse/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y454sZmlsZQ==.jl:2


## Keyword Arguments

  * Functions with keyword arguments are defined using a semicolon in the signature. 
  * Kwargs participate in method dispatch. Methods are dispatched based only on positional arguments, with keyword arguments processed after the matching method is identified. 
  * Under certain conditions I can omit the semicolon when calling a function with keyword arguments. 
  * Keyword arguments can be optional (if they have default values specified) or required (if they don't have default values).
  * `Dict{Symbol, Any}`s are splatted to form kwargs.

In [78]:
function keywords(arg1, arg2; kwarg1=1, kwarg2="kwarg2")
    println("arg1 = $arg1")
    println("arg2 = $arg2")
    println("kwarg1 = $kwarg1")
    println("kwarg2 = $kwarg2")
end

keywords (generic function with 1 method)

In [79]:
keywords(10, 20, kwarg2="hello")

arg1 = 10
arg2 = 20
kwarg1 = 1
kwarg2 = hello


In [80]:
# semicolon before keyword args is optional when passing them
# explicitly
keywords(10, 20; kwarg1=-1, kwarg2="world")

arg1 = 10
arg2 = 20
kwarg1 = -1
kwarg2 = world


In [81]:
kw = Dict(
    :kwarg1 => -1,
    :kwarg2 => "hahaha"
)
# semicolons are required when splatting
keywords(10, 20; kw...)

arg1 = 10
arg2 = 20
kwarg1 = -1
kwarg2 = hahaha


In [82]:
kwarg1 = -10
kwarg2 = "hello"
# the variable names are automatically converted into key-value pairs
# semicolon is required here as well
keywords(10, 20; kwarg1, kwarg2)

arg1 = 10
arg2 = 20
kwarg1 = -10
kwarg2 = hello


In [83]:
# This style is useful if I am generating the kwargs at runtime.
keywords(10, 20; :kwarg1 => -10)

arg1 = 10
arg2 = 20
kwarg1 = -10
kwarg2 = kwarg2


In [84]:
function keywords2(arg1; kwargs...)
    println("arg1 = $arg1")
    for (kwarg, value) in kwargs
        println("$kwarg = $value")
    end
end

keywords2 (generic function with 1 method)

In [85]:
keywords2(1, flavor="Chocolate Chip", calories=200)

arg1 = 1
flavor = Chocolate Chip
calories = 200


In [86]:
cookies = Dict(
    :ChocolateChip => 200,
    :OatmealRaisin => 180
)
keywords2(1; cookies...)

arg1 = 1
OatmealRaisin = 180
ChocolateChip = 200


## Do-blocks

When a higher order function accepts another function as its first argument, I can use the do-block syntax to declare the argument function in an ergonomic fashion.

In [87]:
# without do-block it is awkward to write multiline lambdas
xs = -5:5
map(x -> begin
        if x < 0 && iseven(x)
            return 0
        elseif x == 0
            return 1
        else
            return x
        end
    end,
    xs)

11-element Vector{Int64}:
 -5
  0
 -3
  0
 -1
  1
  1
  2
  3
  4
  5

In [88]:
# with do-block it is more elegant
# map is considered the "outer" function
# do-block will become the first arg passed to the outer function
map(xs) do x
    if x < 0 && iseven(x)
        return 0
    elseif x == 0
        return 1
    else
        return x
    end
end


11-element Vector{Int64}:
 -5
  0
 -3
  0
 -1
  1
  1
  2
  3
  4
  5

## Function Composition

For a composite function $f(g(x))$ -
  * Composition which goes from right to left just like in math - `(f \circ g)(x)`
  * Piping which reads from left to right just like in unix cli - `x |> g |> f`
  * Piping has this other weird syntax where if I have a vector of arguments and a vector of functions, I can use elementwise piping to zip the arguments and the functions.

In [89]:
(sqrt ∘ sum)(1:10)

7.416198487095663

In [90]:
1:10 |> sum |> sqrt

7.416198487095663

In [91]:
# This will call uppercase("a"), reverse("list"), titlecase("of"), length("strings")
# And wrap all the return values into another Vector.
["a", "list", "of", "strings"] .|> [uppercase, reverse, titlecase, length]


4-element Vector{Any}:
  "A"
  "tsil"
  "Of"
 7

## Higher Order Functions

> I'll fill out this section as I learn more higher order functions. 

  * [map](https://docs.julialang.org/en/v1/base/collections/#Base.map) works similar to other languages in that it takes a unary function and a foldable/collection and applies the unary function to each element in the foldable.
  * `map` also works with binary functions (function that takes two arguments) when I provide it with 2 foldables. An element from each foldable is taken to construct the argument list for the given binary function. `map` ends when any of the foldables is exhausted.

In [92]:
function square_or_cube(x)
    if rand() > 0.5
        x = x^2
    else
        x = x^3
    end
    x
end

square_or_cube (generic function with 1 method)

In [93]:
map(square_or_cube, 1:7)

7-element Vector{Int64}:
  1
  8
  9
 16
 25
 36
 49

In [94]:
map(+, [1, 2, 3], [10, 23, 4, 8, 5, 32])

3-element Vector{Int64}:
 11
 25
  7

In [95]:
map(+, rand(2, 3), rand(2, 3))

2×3 Matrix{Float64}:
 0.969771  1.75251   0.520998
 0.516411  0.700833  1.19161

In [96]:
map(+, rand(2, 3), rand(3, 3))

DimensionMismatch: DimensionMismatch: a has size (2, 3), b has size (3, 3), mismatch at dim 1

In [97]:
map(+, 2, rand(2))

1-element Vector{Float64}:
 2.2532410774746943